# BirdCLEF 2026 — Exploratory Data Analysis
Systematic EDA covering:
1. Dataset overview & missing values
2. Class distribution & imbalance
3. Taxonomic group breakdown
4. Audio quality (rating)
5. Geographic distribution
6. Secondary labels / co-occurrence
7. Audio duration analysis
8. Mel spectrogram samples
9. Correlation analysis
10. Feature relevance summary

## 0. Imports & Paths

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")

BASE        = Path("/kaggle/input/birdclef-2026")
TRAIN_AUDIO = BASE / "train_audio"
TRAIN_META  = BASE / "train_metadata.csv"
TAXONOMY    = BASE / "taxonomy.csv"
SAMPLE_SUB  = BASE / "sample_submission.csv"

## 1. Dataset Overview

In [ ]:
meta = pd.read_csv(TRAIN_META)
tax  = pd.read_csv(TAXONOMY)
sub  = pd.read_csv(SAMPLE_SUB)

print("Shape:", meta.shape)
print("\nColumn dtypes:")
print(meta.dtypes)
meta.head(3)

In [ ]:
print("Missing values:")
missing = meta.isnull().sum()
print(missing[missing > 0] if missing.any() else "None")

n_species      = meta["primary_label"].nunique()
scored_species = sub.columns[1:]

print(f"\nTotal training files : {len(meta)}")
print(f"Unique primary labels: {n_species}")
print(f"Scored species       : {len(scored_species)}")
if "class_name" in tax.columns:
    print(f"Taxonomic classes    : {tax['class_name'].nunique()}")

## 2. Class Distribution & Imbalance

In [ ]:
counts = meta["primary_label"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

counts.head(40).plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="none")
axes[0].set(title="Top-40 Most Common Species", xlabel="Species", ylabel="# recordings")
axes[0].tick_params(axis="x", rotation=90, labelsize=7)

axes[1].hist(counts.values, bins=50, color="darkorange", edgecolor="white")
axes[1].set(title="Recordings per Species (distribution)", xlabel="# recordings", ylabel="# species")
axes[1].axvline(counts.median(), color="red", linestyle="--", label=f"Median={counts.median():.0f}")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Min: {counts.min()} | Max: {counts.max()} | Median: {counts.median():.0f}")
print(f"Species with < 5 recordings : {(counts < 5).sum()}")
print(f"Species with > 50 recordings: {(counts > 50).sum()}")

## 3. Taxonomic Group Breakdown

In [ ]:
if "class_name" in tax.columns and "primary_label" in tax.columns:
    meta_tax = meta.merge(
        tax[["primary_label", "class_name", "order", "family"]],
        on="primary_label", how="left")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    meta_tax["class_name"].value_counts().plot(
        kind="bar", ax=axes[0], color="teal", edgecolor="none")
    axes[0].set(title="Recordings by Taxonomic Class", xlabel="Class", ylabel="# recordings")
    axes[0].tick_params(axis="x", rotation=30)

    meta_tax["order"].value_counts().head(20).plot(
        kind="barh", ax=axes[1], color="mediumpurple", edgecolor="none")
    axes[1].set(title="Top-20 Orders", xlabel="# recordings")
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()

    print("Class breakdown:")
    print(meta_tax["class_name"].value_counts())
else:
    print("No class_name column found in taxonomy — check column names:", tax.columns.tolist())

## 4. Audio Quality (Rating)

Ratings are 0–5 (xeno-canto community quality scores). Low-quality recordings may hurt training.

In [ ]:
if "rating" in meta.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    meta["rating"].value_counts().sort_index().plot(
        kind="bar", ax=axes[0], color="seagreen", edgecolor="white")
    axes[0].set(title="Quality Rating Distribution", xlabel="Rating (0–5)", ylabel="# recordings")

    rating_by_species = (meta.groupby("primary_label")["rating"].mean()
                         .reset_index()
                         .rename(columns={"rating": "mean_rating"})
                         .merge(counts.reset_index().rename(columns={"count": "n_recs"}),
                                on="primary_label"))
    axes[1].scatter(rating_by_species["n_recs"], rating_by_species["mean_rating"],
                    alpha=0.5, s=20, color="coral")
    axes[1].set(title="Mean Rating vs. # Recordings per Species",
                xlabel="# recordings", ylabel="Mean rating")

    plt.tight_layout()
    plt.show()

    print(meta["rating"].describe().round(2))
    low = (meta["rating"] < 2).sum()
    print(f"\nLow quality (rating < 2): {low} ({100*low/len(meta):.1f}%)")
else:
    print("No rating column found")

## 5. Geographic Distribution

Useful for building geo-priors in post-processing (species range filtering).

In [ ]:
if {"latitude", "longitude"}.issubset(meta.columns):
    valid_geo = meta.dropna(subset=["latitude", "longitude"])
    print(f"Recordings with coordinates: {len(valid_geo)} / {len(meta)}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].scatter(valid_geo["longitude"], valid_geo["latitude"],
                    alpha=0.3, s=5, c="royalblue")
    axes[0].set(title="Geographic Distribution of Recordings",
                xlabel="Longitude", ylabel="Latitude")

    axes[1].hist(valid_geo["latitude"], bins=60, color="royalblue", edgecolor="none")
    axes[1].set(title="Latitude Distribution", xlabel="Latitude", ylabel="Count")

    plt.tight_layout()
    plt.show()

    print("Lat range:", valid_geo["latitude"].min().round(2), "→", valid_geo["latitude"].max().round(2))
    print("Lon range:", valid_geo["longitude"].min().round(2), "→", valid_geo["longitude"].max().round(2))
else:
    print("No lat/lon columns found")

## 6. Secondary Labels (Co-occurrence)

Secondary labels = other species audible in the same recording. Key for multi-label soft training.

In [ ]:
if "secondary_labels" in meta.columns:
    sec = meta["secondary_labels"].fillna("[]").apply(
        lambda x: eval(x) if isinstance(x, str) else x)
    has_secondary = sec.apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
    print(f"Recordings with secondary labels: {has_secondary.sum()} ({100*has_secondary.mean():.1f}%)")

    all_secondary = [sp for sublist in sec for sp in (sublist if isinstance(sublist, list) else [])]
    top_cooccur = pd.Series(Counter(all_secondary)).sort_values(ascending=False).head(20)

    plt.figure(figsize=(11, 4))
    top_cooccur.plot(kind="bar", color="salmon", edgecolor="none")
    plt.title("Top-20 Secondary Labels (Co-occurring Species)")
    plt.xlabel("Species"); plt.ylabel("# appearances as secondary label")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print("No secondary_labels column found")

## 7. Audio Duration Analysis

All test windows are 5s. Training files vary — need padding (short) or chunking (long).

In [ ]:
sample_files = list(TRAIN_AUDIO.rglob("*.ogg"))[:300]
durations = []
for fp in sample_files:
    try:
        durations.append(librosa.get_duration(path=str(fp)))
    except Exception:
        pass

durations = np.array(durations)
print(f"n={len(durations)} | min={durations.min():.1f}s | max={durations.max():.1f}s | "
      f"mean={durations.mean():.1f}s | median={np.median(durations):.1f}s")
print(f"Recordings < 5s : {(durations < 5).sum()}")
print(f"Recordings > 60s: {(durations > 60).sum()}")

plt.figure(figsize=(10, 4))
plt.hist(durations, bins=60, color="steelblue", edgecolor="white")
plt.axvline(5,  color="red",    linestyle="--", label="5s (min window)")
plt.axvline(60, color="orange", linestyle="--", label="60s")
plt.title("Audio Duration Distribution (sample n=300)")
plt.xlabel("seconds"); plt.legend(); plt.tight_layout(); plt.show()

## 8. Mel Spectrogram Samples

Visual sanity check — clean calls vs noisy backgrounds.

In [ ]:
SR, N_MELS, HOP = 32000, 128, 512
sample_species = meta["primary_label"].value_counts().index[:6].tolist()

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, sp in zip(axes.flat, sample_species):
    files = list((TRAIN_AUDIO / sp).glob("*.ogg"))
    if not files:
        ax.set_title(f"{sp} (no file)"); continue
    y, sr = librosa.load(str(files[0]), sr=SR, duration=5.0)
    mel = librosa.power_to_db(
        librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, hop_length=HOP), ref=np.max)
    librosa.display.specshow(mel, sr=sr, hop_length=HOP,
                             x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(sp, fontsize=9); ax.set_xlabel(""); ax.set_ylabel("")

plt.suptitle("Sample Mel Spectrograms — top-6 species (5s clips)", y=1.01)
plt.tight_layout(); plt.show()

## 9. Correlation Analysis

Checking relationships between numeric metadata features.

In [ ]:
numeric_cols = meta.select_dtypes(include=[np.number]).columns.tolist()
print("Numeric columns:", numeric_cols)

if len(numeric_cols) >= 2:
    corr = meta[numeric_cols].corr()
    plt.figure(figsize=(max(6, len(numeric_cols)), max(5, len(numeric_cols)-1)))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, linewidths=0.5)
    plt.title("Correlation Matrix — Numeric Metadata Features")
    plt.tight_layout(); plt.show()
    print(corr.round(3))
else:
    print("Not enough numeric columns for correlation")

## 10. Feature Relevance Summary

In [ ]:
summary = {
    "Feature": [
        "audio waveform → mel spectrogram",
        "primary_label",
        "secondary_labels",
        "rating",
        "latitude / longitude",
        "date / month",
        "class_name (taxonomy)",
        "author / license / url",
        "filename",
        "collection (xeno-canto vs iNat)",
    ],
    "Use": [
        "PRIMARY model input (image CNN)",
        "PRIMARY training target",
        "Soft multi-label training targets",
        "Sample weighting / quality filtering (drop < 2.0)",
        "Geo-prior post-processing (species range)",
        "Temporal prior (migratory species)",
        "Stratified splits / grouped evaluation",
        "Not useful for modeling",
        "File I/O only",
        "May indicate label noise level",
    ],
    "Priority": [
        "🔴 Critical",
        "🔴 Critical",
        "🟠 High",
        "🟠 High",
        "🟡 Medium (post-proc)",
        "🟡 Medium (post-proc)",
        "🟢 Low",
        "⚪ None",
        "⚪ None",
        "🟢 Low",
    ]
}

pd.DataFrame(summary).style.set_properties(**{"text-align": "left"})